LIBRARIES

In [16]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

INPUTS

In [17]:
IMG_HEIGHT = 224
IMG_WIDTH =224
IMG_CHANNELS= 3
CLASS_NAMES=["lilly", "lotus", "orchid", "sunflower", "tulip"]

In [18]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    r"C:\My Folder\Projects\Computer_Vision\flower_images\train",
    labels='inferred',
    label_mode='int',
    batch_size=16,
    image_size =(IMG_HEIGHT, IMG_WIDTH)
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    r"C:\My Folder\Projects\Computer_Vision\flower_images\val",
    labels='inferred',
    label_mode='int',
    batch_size=16,
    image_size =(IMG_HEIGHT, IMG_WIDTH)
)

normalize = tf.keras.layers.Rescaling(1./255)

train_dataset= train_dataset.map(lambda x, y: (normalize(x), y))
val_dataset = train_dataset.map(lambda x, y: (normalize(x), y))

Found 3000 files belonging to 5 classes.
Found 1000 files belonging to 5 classes.


NETWORK ARCHITECTURE

In [19]:
base_model = tf.keras.applications.VGG16(
    input_shape = (IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
    include_top = False, # To exclude the classifier, linear layers
    weights = "imagenet"
)

base_model.trainable=False

model = keras.Sequential([
    base_model,

    keras.layers.Flatten(),

    keras.layers.Dense(2000, activation='relu'),
    keras.layers.Dropout(0.5),

    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.5),

    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dropout(0.5),

    keras.layers.Dense(len(CLASS_NAMES), activation='softmax')
])

model.compile(
    optimizer = "adam",
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 2000)           │    50,178,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 2000)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 512)            │     1,024,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 100)            │        51,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 5)              │           505 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,969,005 (251.65 MB)

 Trainable params: 51,254,317 (195.52 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

TRAINING

In [20]:
EPOCHS = 10
history = model.fit(
    train_dataset,
    validation_data= val_dataset,
    epochs=EPOCHS
)

Epoch 1/10
 70/188 ━━━━━━━━━━━━━━━━━━━━ 3:43 2s/step - accuracy: 0.2150 - loss: 7.2297

KeyboardInterrupt: 

PLOTTING

In [ ]:
plt.plot(history.history['loss'], label ="Training Loss")
plt.plot(history.history['val_loss'], label = "Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['accuracy'], label ="Training Accuracy")
plt.plot(history.history['val_accuracy'], label = "Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()